In [1]:
import os
from tqdm import tqdm
import re
import sys
import traceback
from flask import Flask, request, jsonify, send_file
from werkzeug.utils import secure_filename
import zipfile
import io
import warnings
import pymysql
from datetime import datetime
warnings.filterwarnings("ignore")

app = Flask(__name__)

# 配置上传文件夹
UPLOAD_FOLDER = 'uploads'
SPLIT_FOLDER = 'split_chapters'
ALLOWED_EXTENSIONS = {'txt'}

# 创建必要的目录
for folder in [UPLOAD_FOLDER, SPLIT_FOLDER]:
    if not os.path.exists(folder):
        os.makedirs(folder)

app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER
app.config['MAX_CONTENT_LENGTH'] = 16 * 1024 * 1024  # 限制上传文件大小为16MB

# 数据库配置
DB_CONFIG = {
    'host': '127.0.0.1',
    'user': 'root',
    'password': '123456',
    'database': 'book_db',
    'charset': 'utf8mb4'
}

def get_db_connection():
    """获取数据库连接"""
    return pymysql.connect(**DB_CONFIG)

def allowed_file(filename):
    return '.' in filename and filename.rsplit('.', 1)[1].lower() in ALLOWED_EXTENSIONS

def get_novel_name(file_path):
    """
    从文件路径中提取小说名称
    
    参数:
        file_path (str): 小说文件路径
    
    返回:
        str: 小说名称
    """
    # 获取文件名（不含扩展名）
    file_name = os.path.basename(file_path)
    novel_name = os.path.splitext(file_name)[0]
    return novel_name

def is_chapter_title(text):
    """
    判断文本是否为章节标题
    
    参数:
        text (str): 待判断的文本
    
    返回:
        tuple: (is_title, title_type)
            - is_title: 是否为章节标题
            - title_type: 标题类型
    """
    # 章节标题检测
    patterns = [
        # 卷章节格式
        r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》',
        r'^第\d+卷\s*《[^》]+》',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^第\d+卷\s*[^（(]+',
        
        # 卷章连体格式
        r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷第\d+章\s*[^（(]+',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷\s*第\d+章\s*[^（(]+',
        
        # 基本章节格式
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+（[一二三四五六七八九十\d]+）',
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^(]+\([一二三四五六七八九十\d]+\)',
        r'^第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+章\s*[^（(]+',
        
        # 回目格式
        r'^第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^第\d+回\s*[^（(]+',
        
        # 特殊章节
        r'^楔子\s*[^（(]*',
        r'^序章\s*[^（(]*',
        r'^尾声\s*[^（(]*',
        
        # 英文格式
        r'^Chapter\s*\d+\s*[^(]*',
        
        # 数字格式
        r'^[一二三四五六七八九十\d]+[、.]\s*[^（(]*',
        r'^\s*[一二三四五六七八九十\d]+[、.]\s*[^（(]*',
        
        # 带引号的格式
        r'^『[一二三四五六七八九十\d]+』\s*[^（(]*',
        r'^「[一二三四五六七八九十\d]+」\s*[^（(]*',
        r'^《[一二三四五六七八九十\d]+》\s*[^（(]*',
        
        # 带括号的格式
        r'^（[一二三四五六七八九十\d]+）\s*[^（(]*',
        r'^\([一二三四五六七八九十\d]+\)\s*[^（(]*',
        
        # 其他常见格式
        r'^\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^\s*第\d+章\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+回\s*[^（(]+',
        r'^\s*第\d+回\s*[^（(]+',
    ]
    
    # 检查是否是卷标题
    volume_patterns = [
        r'^第[一二三四五六七八九十百千万\d]+卷\s*《[^》]+》',
        r'^第\d+卷\s*《[^》]+》',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^第\d+卷\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+卷\s*[^（(]+',
        r'^\s*第\d+卷\s*[^（(]+',
    ]
    
    # 检查是否是卷章连体格式
    volume_chapter_patterns = [
        r'^第[一二三四五六七八九十百千万\d]+卷第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷第\d+章\s*[^（(]+',
        r'^第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^第\d+卷\s*第\d+章\s*[^（(]+',
        r'^\s*第[一二三四五六七八九十百千万\d]+卷\s*第[一二三四五六七八九十百千万\d]+章\s*[^（(]+',
        r'^\s*第\d+卷\s*第\d+章\s*[^（(]+',
    ]
    
    # 首先检查是否是卷章连体格式
    for pattern in volume_chapter_patterns:
        if re.match(pattern, text):
            return True, 'volume_chapter'
    
    # 然后检查是否是卷标题
    for pattern in volume_patterns:
        if re.match(pattern, text):
            return True, 'volume'
    
    # 最后检查其他格式
    for pattern in patterns:
        if re.match(pattern, text):
            return True, 'chapter'
    
    return False, 'other'

def save_book_to_db(book_name, chapter_count):
    """保存书籍信息到数据库"""
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                sql = """
                INSERT INTO book (book_name, chapter_count, create_time, update_time)
                VALUES (%s, %s, %s, %s)
                """
                cursor.execute(sql, (book_name, chapter_count, now, now))
                conn.commit()
                return cursor.lastrowid
    except Exception as e:
        print(f"保存书籍信息时出错: {e}")
        raise

def save_chapter_to_db(book_id, chapter_number, chapter_title, content_path, word_count):
    """保存章节信息到数据库"""
    try:
        with get_db_connection() as conn:
            with conn.cursor() as cursor:
                now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                sql = """
                INSERT INTO book_chapter (book_id, chapter_number, chapter_title, 
                                        content_path, word_count, create_time, update_time)
                VALUES (%s, %s, %s, %s, %s, %s, %s)
                """
                cursor.execute(sql, (book_id, chapter_number, chapter_title, 
                                   content_path, word_count, now, now))
                conn.commit()
    except Exception as e:
        print(f"保存章节信息时出错: {e}")
        raise

def split_novel(novel_path, base_output_dir='split_chapters'):
    """
    将小说分割成单独的章节文件并保存到数据库
    
    参数:
        novel_path (str): 小说文件路径
        base_output_dir (str): 基础输出目录
    
    返回:
        tuple: (success, message, book_id)
    """
    try:
        if not os.path.exists(novel_path):
            return False, f"错误：小说文件不存在 - {novel_path}", None
            
        novel_name = get_novel_name(novel_path)
        print(f"处理小说: {novel_name}")
        
        output_dir = os.path.join(base_output_dir, novel_name)
        os.makedirs(output_dir, exist_ok=True)
        
        print("正在读取小说内容...")
        with open(novel_path, 'r', encoding='utf-8') as f:
            lines = f.readlines()
        
        print(f"共读取到 {len(lines)} 行内容")
        
        current_volume = None
        current_chapter = None
        current_content = []
        chapters = []
        chapter_number = 0
        
        for line in tqdm(lines, desc="处理小说内容"):
            line = line.strip()
            if not line:
                continue
            
            is_title, title_type = is_chapter_title(line)
            
            if is_title:
                if current_chapter is not None and current_content:
                    chapter_number += 1
                    content_text = '\n'.join(current_content)
                    word_count = len(content_text)
                    
                    # 创建章节文件
                    if title_type == 'volume_chapter':
                        chapter_file = f"{current_chapter}.txt"
                    elif current_volume:
                        chapter_file = f"{current_volume}_{current_chapter}.txt"
                    else:
                        chapter_file = f"{current_chapter}.txt"
                    
                    file_path = os.path.join(output_dir, chapter_file)
                    with open(file_path, 'w', encoding='utf-8') as f:
                        f.write(content_text)
                    
                    chapters.append({
                        'number': chapter_number,
                        'title': current_chapter,
                        'path': file_path,
                        'word_count': word_count
                    })
                    print(f"已保存章节: {current_chapter}")
                
                if title_type == 'volume':
                    current_volume = line
                    current_chapter = None
                    current_content = []
                elif title_type == 'volume_chapter':
                    current_chapter = line
                    current_content = []
                else:
                    current_chapter = line
                    current_content = []
            else:
                current_content.append(line)
        
        # 处理最后一个章节
        if current_chapter is not None and current_content:
            chapter_number += 1
            content_text = '\n'.join(current_content)
            word_count = len(content_text)
            
            if title_type == 'volume_chapter':
                chapter_file = f"{current_chapter}.txt"
            elif current_volume:
                chapter_file = f"{current_volume}_{current_chapter}.txt"
            else:
                chapter_file = f"{current_chapter}.txt"
            
            file_path = os.path.join(output_dir, chapter_file)
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(content_text)
            
            chapters.append({
                'number': chapter_number,
                'title': current_chapter,
                'path': file_path,
                'word_count': word_count
            })
            print(f"已保存最后一个章节: {current_chapter}")
        
        # 保存到数据库
        print("正在保存到数据库...")
        book_id = save_book_to_db(novel_name, len(chapters))
        
        for chapter in chapters:
            save_chapter_to_db(
                book_id=book_id,
                chapter_number=chapter['number'],
                chapter_title=chapter['title'],
                content_path=chapter['path'],
                word_count=chapter['word_count']
            )
        
        return True, f"成功处理完成，共分割{len(chapters)}个章节，书籍ID: {book_id}", book_id
        
    except Exception as e:
        error_details = traceback.format_exc()
        print(f"处理过程中出错: {error_details}")
        return False, f"处理过程中出错: {str(e)}", None

@app.route('/split', methods=['POST'])
def split_novel_api():
    """处理小说分章API端点"""
    try:
        if 'file' not in request.files:
            return jsonify({'success': False, 'message': '没有上传文件'}), 400
        
        file = request.files['file']
        if file.filename == '':
            return jsonify({'success': False, 'message': '没有选择文件'}), 400
        
        if not allowed_file(file.filename):
            return jsonify({'success': False, 'message': '不支持的文件类型'}), 400
        
        filename = secure_filename(file.filename)
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        print(f"接收到文件: {filename}")
        
        file.save(filepath)
        print("文件保存成功，开始处理...")
        
        success, message, book_id = split_novel(filepath)
        
        # 清理临时文件
        if os.path.exists(filepath):
            os.remove(filepath)
            print("临时文件已清理")
        
        if not success:
            return jsonify({
                'success': False,
                'message': message
            }), 500
        
        return jsonify({
            'success': True,
            'message': message,
            'book_id': book_id
        })
        
    except Exception as e:
        error_details = traceback.format_exc()
        print(f"错误详情: {error_details}")
        
        if 'filepath' in locals() and os.path.exists(filepath):
            os.remove(filepath)
            print("发生错误，临时文件已清理")
        
        return jsonify({
            'success': False,
            'message': str(e),
            'details': error_details
        }), 500

if __name__ == '__main__':
    print("服务启动中...")
    print("正在连接数据库...")
    
    # 测试数据库连接
    try:
        with get_db_connection() as conn:
            print("数据库连接成功！")
    except Exception as e:
        print(f"数据库连接失败: {e}")
        sys.exit(1)
    
    # 启动服务器
    port = 5000
    try:
        app.run(host='0.0.0.0', port=port)
    except Exception as e:
        print(f"启动服务时出错: {str(e)}")
        sys.exit(1) 

服务启动中...
正在连接数据库...
数据库连接成功！
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5000
 * Running on http://192.168.1.101:5000
Press CTRL+C to quit
